In [0]:
#initial commit

In [0]:
import mlflow
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col, count, when

In [0]:
df = spark.sql("SELECT * FROM analysis.ml_features")

df.show()

In [0]:
features = [
    "payment_value",
    "payment_installments",
    "actual_delivery_days",
    "estimated_delivery_days"
]

In [0]:
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

In [0]:
assembler = VectorAssembler(
    inputCols=features,
    outputCol="features",
    handleInvalid="skip"
)

data = assembler.transform(df)

In [0]:
train, test = data.randomSplit([0.8, 0.2], seed=42)

In [0]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="is_late"
)

model = lr.fit(train)

In [0]:
predictions = model.transform(test)

predictions.select(
    "is_late",
    "prediction",
    "probability"
).show()

In [0]:
evaluator = BinaryClassificationEvaluator(
    labelCol="is_late"
)

accuracy = evaluator.evaluate(predictions)

print("Model Accuracy:", accuracy)

In [0]:
with mlflow.start_run():
    mlflow.log_param("model", "logistic_regression")
    mlflow.log_metric("accuracy", accuracy)

In [0]:
with mlflow.start_run(run_name="Late_Delivery_Model_Spark") as run:

    # 1. Predictions
    predictions = model.transform(test)

    # 2. Evaluate
    evaluator = BinaryClassificationEvaluator(
        labelCol="is_late",
        metricName="areaUnderROC"
    )
    
    auc = evaluator.evaluate(predictions)

    # 3. Log metrics
    mlflow.log_metric("AUC", auc)

    # 4. Log parameters
    mlflow.log_param("model_type", "LogisticRegression")

    # Example extra params (VERY GOOD FOR PROJECT)
    mlflow.log_param("features", "payment_value, freight_value, product_weight, installments")

    # 5. Print results
    print(f"✅ Run completed")
    print(f"📊 AUC: {auc}")
    print(f"🆔 Run ID: {run.info.run_id}")

In [0]:
print(predictions.columns)

In [0]:
predictions.select(
    "order_id",
    "customer_id",
    "prediction",
    "probability"
).write.mode("overwrite").saveAsTable("analysis.predicted_late_deliveries")